# 06 - AHP Priority Ranking Design

## Decision Support Stage

This notebook documents the Phase 10A AHP criteria and expert judgement framework for SentiRank. It does not calculate AHP weights, consistency ratio, priority scores, or final rankings in this phase.


## Phase 10B Backend Availability

Backend AHP calculation is available through `POST /ahp/calculate`. This notebook still does not run final expert judgement calculation; final AHP weights should be generated only after validated expert inputs are available.


## AHP Role In SentiRank

AHP will convert expert pairwise judgements into priority weights for Spotify improvement criteria. The criteria come from the selected SVM `merged_5class` aspect taxonomy and still require expert validation before weighting.


In [ ]:
from pathlib import Path
import json

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "ml-service").exists()
            and (candidate / "docs").exists()
            and (candidate / "datasets").exists()
        ):
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root()
TEMPLATE_DIR = PROJECT_ROOT / "docs" / "templates" / "ahp"
CRITERIA_JSON = TEMPLATE_DIR / "final_criteria_for_ahp.json"
AHP_TEMPLATE_CSV = TEMPLATE_DIR / "ahp_pairwise_template.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TEMPLATE_DIR:", TEMPLATE_DIR)


def load_json(path: Path):
    if not path.exists():
        print(f"Missing JSON: {path.relative_to(PROJECT_ROOT)}")
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing CSV: {path.relative_to(PROJECT_ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path)


## Final Five Criteria

The final candidate criteria are:

1. Features, Content & Audio Experience
2. App Reliability & Usability
3. Ads Experience
4. Subscription & Pricing
5. Account/Login

`General` is excluded because it is a weak-label fallback and not an actionable AHP criterion.


In [ ]:
criteria_payload = load_json(CRITERIA_JSON)
if criteria_payload:
    criteria_df = pd.DataFrame(criteria_payload.get("criteria", []))
    display(criteria_df)


## AHP Pairwise Comparison Design

With 5 criteria, the expert judgement template contains 10 pairwise comparisons. The template pre-fills only `comparison_id`, `criterion_a`, and `criterion_b`; expert answer fields remain blank.


In [ ]:
ahp_template_df = load_csv(AHP_TEMPLATE_CSV)
if not ahp_template_df.empty:
    print("Pairwise comparisons:", len(ahp_template_df))
    display(ahp_template_df)


## Saaty Scale And Consistency Requirement

AHP will use Saaty's 1-9 scale: 1 equal, 3 moderate, 5 strong, 7 very strong, 9 extreme, with 2/4/6/8 as intermediate values. Later AHP calculation must check the consistency ratio. Acceptable consistency is `CR <= 0.10`. If `CR > 0.10`, the judgement should be revised by the expert or documented and excluded/flagged as inconsistent.


## Expected Later Output

Later phases may create AHP weights, consistency-ratio reports, and priority rankings. Phase 10A creates only criteria definitions and expert judgement templates.
